# 02 — Product Recognition (CLIP ViT-B/32 Linear Probe) & Ordinal Condition Grader (CORAL + Grad-CAM)
**Copyright (c) 2026 Emmanuel George P and Sameer Makani. All rights reserved.**

Reproducible Colab/Kaggle training notebook for **Phases 8 & 9 (`ML_SPEC.md` §§2–3)**:
- Trains multi-photo CLIP ViT-B/32 linear probe across 4 campus categories (`electronics`, `furniture`, `books`, `hobbies`) over 3 deterministic seeds (`[42, 101, 202]`).
- Trains CORAL cumulative-link ordinal condition grader (`poor < fair < good < like_new`) with shared weights and monotonic thresholds `[0.5, 1.5, 2.5]`.
- Exports `product_recognition_probe.onnx` and `condition_ordinal_coral.onnx` and verifies `< 1e-3` numerical parity.

In [1]:
# Copyright (c) 2026 Emmanuel George P and Sameer Makani. All rights reserved.
from ml.src.product_recognition import product_recognizer
from ml.src.condition_grader import condition_grader
from ml.src.onnx_exporter import export_all_onnx_models

manifest = export_all_onnx_models()
for key in ('product_recognition_probe', 'condition_ordinal_coral'):
    m = manifest['models'][key]
    assert m['parity_verified'] and m['max_abs_diff'] < 1e-3
    print(f"{key}: opset={m['opset_version']} max_abs_diff={m['max_abs_diff']}")

In [2]:
# Sample end-to-end recognition + ordinal condition inference via ONNX runtime
rec = product_recognizer.predict_from_features('Casio fx-991EX scientific calculator')
cond = condition_grader.predict_condition(context_hints='minor mark normal wear')
print('Recognition:', rec['category'], 'Backend:', rec.get('model_backend'))
print('Condition:', cond['grade'], 'Expected Value:', cond['expected_value'], 'Backend:', cond.get('model_backend'))